# Chobe Capstone: A* Reproducibility Study

This notebook implements and evaluates three search algorithms on randomly generated Gridworld environments:

- **Uniform Cost Search (UCS)**
- **Standard A\*** using the Manhattan distance heuristic
- **Weighted A\*** using a scaled heuristic

The main idea is to compare these algorithms on:

- **path cost**
- **nodes expanded**
- **runtime**
- **peak frontier size**

We test them on easy, medium, and hard environments to study the tradeoff between **optimality** and **efficiency**.


## 1. Imports and Result Data Structure

This section imports the libraries used throughout the notebook and defines a `SearchResult` data class.

### Why this section matters
- `heapq` is used to implement the priority queue.
- `random` gives us reproducible obstacle generation using seeds.
- `time` helps measure runtime.
- `statistics` helps summarize benchmark results.
- `SearchResult` stores all important metrics from one run in a clean structured format.


In [ ]:
import csv
import heapq
import math
import random
import statistics
import time
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

# A grid position is represented as (row, column).
Position = Tuple[int, int]

@dataclass
class SearchResult:
    # Name of the algorithm used in the experiment.
    algorithm: str

    # Size of the grid.
    grid_rows: int
    grid_cols: int

    # Fraction of blocked cells in the generated environment.
    obstacle_density: float

    # Random seed used to generate the grid.
    seed: int

    # Heuristic weight. For UCS and standard A*, this stays at 1.0.
    weight: float

    # Cost of the final path. This is None if no solution exists.
    path_cost: Optional[int]

    # Number of nodes removed from the frontier and expanded.
    nodes_expanded: int

    # Total runtime in milliseconds.
    runtime_ms: float

    # Largest number of nodes stored in the frontier at any time.
    peak_frontier_size: int

    # Number of states in the path itself.
    path_length: int

    # Whether the goal was successfully reached.
    solved: bool


## 2. Priority Queue and Basic Helper Functions

This section defines the low-level tools used by the search algorithm.

### What these helpers do
- `PriorityQueue`: always returns the node with the **lowest priority**
- `manhattan(...)`: heuristic estimate from current node to goal
- `in_bounds(...)`: checks whether a cell is inside the grid
- `get_neighbors(...)`: returns valid up/down/left/right moves
- `reconstruct_path(...)`: rebuilds the final path by following parent pointers backward


In [ ]:
class PriorityQueue:
    def __init__(self) -> None:
        # Heap stores tuples of (priority, tie_breaker, position).
        # The tie_breaker makes ordering stable when priorities are equal.
        self.heap: List[Tuple[float, int, Position]] = []
        self.counter = 0

    def push(self, priority: float, item: Position) -> None:
        # Insert a node into the min-heap.
        heapq.heappush(self.heap, (priority, self.counter, item))
        self.counter += 1

    def pop(self) -> Tuple[float, Position]:
        # Remove and return the node with minimum priority.
        priority, _, item = heapq.heappop(self.heap)
        return priority, item

    def __len__(self) -> int:
        return len(self.heap)


def manhattan(a: Position, b: Position) -> int:
    # Manhattan distance is admissible for 4-direction grid movement.
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def in_bounds(pos: Position, rows: int, cols: int) -> bool:
    # Check whether a position lies inside the grid.
    r, c = pos
    return 0 <= r < rows and 0 <= c < cols


def get_neighbors(pos: Position, rows: int, cols: int, blocked: set) -> List[Position]:
    # Generate the four legal moves from the current position.
    r, c = pos
    candidates = [(r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)]
    neighbors = []

    for nxt in candidates:
        # Keep only positions that are inside the grid and not blocked.
        if in_bounds(nxt, rows, cols) and nxt not in blocked:
            neighbors.append(nxt)

    return neighbors


def reconstruct_path(parent: Dict[Position, Optional[Position]], goal: Position) -> List[Position]:
    # Reconstruct the path from goal back to start using parent pointers.
    path = []
    cur = goal

    while cur is not None:
        path.append(cur)
        cur = parent[cur]

    # Reverse so the path goes start -> goal.
    path.reverse()
    return path


## 3. Grid Generation and Visualization

This section creates the Gridworld environment and prints it in a readable text form.

### Grid symbols
- `S` = start
- `G` = goal
- `#` = blocked cell / obstacle
- `.` = free cell
- `*` = path found by the algorithm

Using a random seed ensures that the same grid can be regenerated later, which is important for reproducibility.


In [ ]:
def generate_grid(
    rows: int,
    cols: int,
    obstacle_density: float,
    seed: int,
    start: Position,
    goal: Position
) -> set:
    # Use a local random generator so results are reproducible for the same seed.
    rng = random.Random(seed)
    blocked = set()

    for r in range(rows):
        for c in range(cols):
            cell = (r, c)

            # Never block the start or goal cell.
            if cell == start or cell == goal:
                continue

            # Randomly mark the cell as blocked based on the chosen density.
            if rng.random() < obstacle_density:
                blocked.add(cell)

    return blocked


def print_grid(
    rows: int,
    cols: int,
    blocked: set,
    start: Position,
    goal: Position,
    path: Optional[List[Position]] = None
) -> None:
    # Convert the path to a set for fast lookup during printing.
    path_set = set(path) if path else set()

    for r in range(rows):
        line = []
        for c in range(cols):
            cell = (r, c)

            if cell == start:
                line.append("S")
            elif cell == goal:
                line.append("G")
            elif cell in path_set:
                line.append("*")
            elif cell in blocked:
                line.append("#")
            else:
                line.append(".")

        print(" ".join(line))

    print()


## 4. Unified Search Engine: UCS, A\*, and Weighted A\*

This is the core of the notebook.

### Main idea
All three algorithms are implemented inside **one** function.  
The only difference is how the priority is computed:

- **UCS:** `priority = g(n)`
- **A\*:** `priority = g(n) + h(n)`
- **Weighted A\*:** `priority = g(n) + weight * h(n)`

### Important data structures
- `frontier`: priority queue of nodes to explore next
- `g_cost`: best known cost from start to each node
- `parent`: used for path reconstruction
- `closed`: nodes already expanded


In [ ]:
def best_first_search(
    rows: int,
    cols: int,
    blocked: set,
    start: Position,
    goal: Position,
    algorithm_name: str,
    weight: float = 1.0
) -> Tuple[Optional[List[Position]], int, int]:
    # Initialize the priority queue with the start node.
    frontier = PriorityQueue()
    frontier.push(0.0, start)

    # g_cost stores the best known distance from start to each node.
    g_cost: Dict[Position, float] = {start: 0.0}

    # parent stores the predecessor of each node for path reconstruction.
    parent: Dict[Position, Optional[Position]] = {start: None}

    # closed contains nodes that have already been expanded.
    closed = set()

    # Metrics collected during search.
    nodes_expanded = 0
    peak_frontier_size = 1

    # Continue until there are no more nodes left to explore.
    while len(frontier) > 0:
        _, current = frontier.pop()

        # Skip stale entries that may still exist in the heap.
        if current in closed:
            continue

        # Mark current node as expanded.
        closed.add(current)
        nodes_expanded += 1

        # If goal is reached, reconstruct and return the solution path.
        if current == goal:
            return reconstruct_path(parent, goal), nodes_expanded, peak_frontier_size

        # Explore all valid neighbors.
        for neighbor in get_neighbors(current, rows, cols, blocked):
            # Each move has unit cost.
            tentative_g = g_cost[current] + 1.0

            # Update only if this path to the neighbor is better than any previous one.
            if neighbor not in g_cost or tentative_g < g_cost[neighbor]:
                g_cost[neighbor] = tentative_g
                parent[neighbor] = current

                # Priority depends on which algorithm is being used.
                if algorithm_name == "UCS":
                    # UCS ignores the heuristic.
                    priority = tentative_g
                elif algorithm_name == "A*":
                    # A* combines path cost so far and heuristic estimate.
                    priority = tentative_g + float(manhattan(neighbor, goal))
                elif algorithm_name == "Weighted A*":
                    # Weighted A* puts more trust in the heuristic.
                    priority = tentative_g + weight * float(manhattan(neighbor, goal))
                else:
                    raise ValueError(f"Unknown algorithm: {algorithm_name}")

                # Push updated neighbor into the frontier.
                frontier.push(priority, neighbor)
                peak_frontier_size = max(peak_frontier_size, len(frontier))

    # If the loop ends, no path exists.
    return None, nodes_expanded, peak_frontier_size


## 5. Running a Single Experiment and Benchmark Suite

This section wraps the search algorithm into experiment utilities.

### `run_single_experiment(...)`
Runs one algorithm on one grid and records:
- solution path
- nodes expanded
- runtime
- frontier size
- whether the grid was solved

### `run_benchmark(...)`
Runs many seeds and algorithms, then saves results to CSV.

### `summarize_results(...)`
Groups results by algorithm and weight and prints averages.


In [ ]:
def run_single_experiment(
    rows: int,
    cols: int,
    obstacle_density: float,
    seed: int,
    algorithm_name: str,
    weight: float,
    show_grid: bool = False
) -> SearchResult:
    # Define fixed start and goal positions.
    start = (0, 0)
    goal = (rows - 1, cols - 1)

    # Generate the blocked cells for this experiment.
    blocked = generate_grid(rows, cols, obstacle_density, seed, start, goal)

    # Time the search execution.
    t0 = time.perf_counter()
    path, nodes_expanded, peak_frontier_size = best_first_search(
        rows=rows,
        cols=cols,
        blocked=blocked,
        start=start,
        goal=goal,
        algorithm_name=algorithm_name,
        weight=weight
    )
    t1 = time.perf_counter()

    runtime_ms = (t1 - t0) * 1000.0

    # Convert path information into readable metrics.
    if path is None:
        path_cost = None
        path_length = 0
        solved = False
    else:
        path_cost = len(path) - 1
        path_length = len(path)
        solved = True

    # Optionally print the grid and the discovered path.
    if show_grid:
        print(f"Algorithm: {algorithm_name}, weight={weight}, seed={seed}")
        print_grid(rows, cols, blocked, start, goal, path)

    # Return all metrics in one structured result object.
    return SearchResult(
        algorithm=algorithm_name,
        grid_rows=rows,
        grid_cols=cols,
        obstacle_density=obstacle_density,
        seed=seed,
        weight=weight,
        path_cost=path_cost,
        nodes_expanded=nodes_expanded,
        runtime_ms=runtime_ms,
        peak_frontier_size=peak_frontier_size,
        path_length=path_length,
        solved=solved
    )


def run_benchmark(
    rows: int,
    cols: int,
    obstacle_density: float,
    seeds: List[int],
    weights: List[float],
    output_csv: str = "results.csv"
) -> List[SearchResult]:
    # Store every experiment result in a single list.
    results: List[SearchResult] = []

    # Run UCS and A* once per seed, and Weighted A* once for each chosen weight.
    for seed in seeds:
        results.append(run_single_experiment(rows, cols, obstacle_density, seed, "UCS", 1.0))
        results.append(run_single_experiment(rows, cols, obstacle_density, seed, "A*", 1.0))

        for w in weights:
            results.append(run_single_experiment(rows, cols, obstacle_density, seed, "Weighted A*", w))

    # Save raw experiment data to CSV for later inspection.
    with open(output_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "algorithm", "grid_rows", "grid_cols", "obstacle_density", "seed", "weight",
            "path_cost", "nodes_expanded", "runtime_ms", "peak_frontier_size",
            "path_length", "solved"
        ])

        for r in results:
            writer.writerow([
                r.algorithm, r.grid_rows, r.grid_cols, r.obstacle_density, r.seed, r.weight,
                r.path_cost, r.nodes_expanded, f"{r.runtime_ms:.4f}", r.peak_frontier_size,
                r.path_length, r.solved
            ])

    return results


def summarize_results(results: List[SearchResult]) -> None:
    # Group results by (algorithm, weight) so averages can be computed.
    grouped: Dict[Tuple[str, float], List[SearchResult]] = {}

    for r in results:
        key = (r.algorithm, r.weight)
        grouped.setdefault(key, []).append(r)

    print("\n===== SUMMARY =====")
    print(
        f"{'Algorithm':<15} {'Weight':<8} {'Solved':<8} {'AvgCost':<10} "
        f"{'AvgExpanded':<14} {'AvgRuntime(ms)':<16} {'AvgPeakFrontier':<16}"
    )

    # Compute summary statistics for each algorithm configuration.
    for key, group in sorted(grouped.items(), key=lambda x: (x[0][0], x[0][1])):
        solved_runs = [g for g in group if g.solved]
        solved_count = len(solved_runs)

        avg_cost = (
            statistics.mean(g.path_cost for g in solved_runs if g.path_cost is not None)
            if solved_runs else math.nan
        )
        avg_expanded = statistics.mean(g.nodes_expanded for g in group)
        avg_runtime = statistics.mean(g.runtime_ms for g in group)
        avg_frontier = statistics.mean(g.peak_frontier_size for g in group)

        avg_cost_str = f"{avg_cost:.2f}" if solved_runs else "N/A"

        print(
            f"{key[0]:<15} {key[1]:<8.2f} {solved_count:<8} {avg_cost_str:<10} "
            f"{avg_expanded:<14.2f} {avg_runtime:<16.4f} {avg_frontier:<16.2f}"
        )


## 6. Single Demo Run

This cell runs one small example that is easy to inspect visually.

### Why this is useful
It helps verify that:
- the grid is generated correctly
- the algorithms run on the same environment
- the path or failure case is easy to inspect before large benchmarks


In [ ]:
# Small demo configuration for visual inspection.
rows = 12
cols = 12
density = 0.20
seed = 7
weights = [1.5, 2.0, 3.0]

print("===== SINGLE DEMO RUN =====\n")

# Run UCS and standard A* on the same grid.
ucs_result = run_single_experiment(rows, cols, density, seed, "UCS", 1.0, show_grid=True)
astar_result = run_single_experiment(rows, cols, density, seed, "A*", 1.0, show_grid=True)

print(ucs_result)
print(astar_result)

# Run Weighted A* for several heuristic weights.
for w in weights:
    wa_result = run_single_experiment(rows, cols, density, seed, "Weighted A*", w, show_grid=True)
    print(wa_result)


## 7. Medium Benchmark (20x20, Density 0.20)

This is the main benchmark used to compare the algorithms on a moderate-difficulty environment.

### Setup
- grid size: 20x20
- obstacle density: 0.20
- seeds: 1 through 10
- Weighted A* weights: 1.5, 2.0, 3.0

The output CSV is saved so the raw numbers can be reused later in the report.


In [ ]:
# Benchmark on a medium-difficulty environment.
seeds = list(range(1, 11))
weights = [1.5, 2.0, 3.0]

results = run_benchmark(
    rows=20,
    cols=20,
    obstacle_density=0.20,
    seeds=seeds,
    weights=weights,
    output_csv="results_20x20_d020.csv"
)

summarize_results(results)
print("Detailed results written to results_20x20_d020.csv")


## 8. Easy Benchmark (15x15, Density 0.10)

This benchmark represents an easier search environment.

It is useful for checking whether:
- A* preserves optimality relative to UCS
- Weighted A* gains speed with only a small loss in path quality


In [ ]:
# Benchmark on an easier environment.
results1 = run_benchmark(
    rows=15,
    cols=15,
    obstacle_density=0.10,
    seeds=list(range(1, 11)),
    weights=[1.5, 2.0, 3.0],
    output_csv="results_15x15_d010.csv"
)
summarize_results(results1)


## 9. Hard Benchmark (30x30, Density 0.25)

This benchmark represents a harder environment with denser obstacles and lower solvability.

This is especially useful for observing:
- how much heuristic guidance helps
- whether Weighted A* still improves efficiency
- how solution quality changes in harder settings


In [ ]:
# Benchmark on a harder environment.
results3 = run_benchmark(
    rows=30,
    cols=30,
    obstacle_density=0.25,
    seeds=list(range(1, 11)),
    weights=[1.5, 2.0, 3.0],
    output_csv="results_30x30_d025.csv"
)
summarize_results(results3)


## 10. Optional Re-run Cells

These cells are optional quick reruns.  
They repeat the easy and hard benchmarks without explicitly writing output CSV files.

You can keep them for convenience or remove them later to make the notebook shorter.


In [ ]:
# Optional quick re-run of the easy benchmark.
results1 = run_benchmark(
    rows=15,
    cols=15,
    obstacle_density=0.10,
    seeds=list(range(1, 11)),
    weights=[1.5, 2.0, 3.0]
)
summarize_results(results1)


In [ ]:
# Optional quick re-run of the hard benchmark.
results3 = run_benchmark(
    rows=30,
    cols=30,
    obstacle_density=0.25,
    seeds=list(range(1, 11)),
    weights=[1.5, 2.0, 3.0]
)
summarize_results(results3)
